## Install libraries

In [ ]:
# Optional: install XAI libraries if they are not already in the environment.
# Leave as False during normal notebook runs to avoid reinstalling packages every time.
INSTALL_MISSING_PACKAGES = False

if INSTALL_MISSING_PACKAGES:
    import subprocess
    subprocess.run(["pip", "install", "captum", "shap", "lime"], check=True)


## Imports and config

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import transforms, models
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# XAI
from captum.attr import LayerGradCam, LayerAttribution
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths — notebook is assumed to run from src/
BASE_DIR = Path("..")

HAM_IMG_DIR = BASE_DIR / "data" / "ham10000" / "images"
HAM_CSV = BASE_DIR / "data" / "preprocessed_manifests" / "ham10000_preprocessed.csv"

ISIC_IMG_DIR = BASE_DIR / "data" / "isic2018" / "images"

PILOT_CSV = BASE_DIR / "data" / "pilot_100" / "pilot_subset_100.csv"

CHECKPOINT_DIR = Path("checkpoints")

# Pilot-specific XAI output folder

XAI_OUT_DIR = BASE_DIR / "outputs" / "pilot_100" / "xai_maps"
XAI_PREVIEW_DIR = BASE_DIR / "outputs" / "pilot_100" / "xai_previews"
XAI_MANIFEST_DIR = BASE_DIR / "outputs" / "pilot_100" / "manifests"

for d in [XAI_OUT_DIR, XAI_PREVIEW_DIR, XAI_MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for method in ["gradcam", "shap", "lime"]:
    (XAI_OUT_DIR / method).mkdir(parents=True, exist_ok=True)

# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")
print("Pilot CSV:", PILOT_CSV.resolve())
print("XAI output:", XAI_OUT_DIR.resolve())


## Rebuild model and load checkpoint

In [ ]:
# Rebuild the exact same architecture as Phase 2
model = models.resnet50(weights=None)

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 1)
)

checkpoint = torch.load(
    CHECKPOINT_DIR / "best_resnet50_finetuned.pt",
    map_location=DEVICE
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"Checkpoint loaded — val_acc at save: {checkpoint['val_acc']:.4f}")

## Load data and build val split (same as Phase 2)

In [ ]:
from sklearn.model_selection import train_test_split

assert PILOT_CSV.exists(), f"Missing pilot subset: {PILOT_CSV.resolve()}"

pilot_df = pd.read_csv(PILOT_CSV).copy()

# Normalise dataset names used across notebooks.
pilot_df["dataset"] = pilot_df["dataset"].astype(str).str.lower()

DATASET_ALIASES = {
    "ham": "ham10000",
    "ham10000": "ham10000",
    "isic": "isic2018",
    "isic2018": "isic2018",
}
pilot_df["dataset"] = pilot_df["dataset"].map(DATASET_ALIASES).fillna(pilot_df["dataset"])

if "image_exists" in pilot_df.columns:
    pilot_df = pilot_df[pilot_df["image_exists"].astype(bool)].copy()

pilot_df = pilot_df.reset_index(drop=True)

def resolve_path(path_value, base_dir: Path = BASE_DIR) -> Path | None:
    """Resolve a path that may be absolute or relative to the repository root."""
    if path_value is None or pd.isna(path_value):
        return None

    p = Path(str(path_value))

    if p.is_absolute() and p.exists():
        return p

    candidates = [
        p,
        base_dir / p,
        Path.cwd() / p,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None

def get_image_path(row: pd.Series) -> Path:
    """Return the image path for either HAM10000 or ISIC2018 pilot rows."""
    for col in ["image_path", "img_path"]:
        if col in row.index:
            resolved = resolve_path(row[col])
            if resolved is not None:
                return resolved

    stem = str(row["stem"])
    dataset = str(row["dataset"]).lower()

    if dataset == "ham10000":
        return HAM_IMG_DIR / f"{stem}.jpg"

    if dataset == "isic2018":
        return ISIC_IMG_DIR / f"{stem}.jpg"

    raise ValueError(f"Unknown dataset for row {stem}: {dataset}")

samples_df = pilot_df.copy()

# Keep these columns available even for ISIC rows where diagnostic labels do not exist.
for col, default in {
    "dx": "unknown",
    "label_name": "unknown",
    "label": np.nan,
}.items():
    if col not in samples_df.columns:
        samples_df[col] = default
    else:
        samples_df[col] = samples_df[col].fillna(default)

samples_df["image_resolved_path"] = samples_df.apply(lambda r: str(get_image_path(r)), axis=1)

print(f"Pilot samples selected: {len(samples_df)}")
print(samples_df["dataset"].value_counts())
display(samples_df[["dataset", "stem", "dx", "label_name", "image_resolved_path"]].head())

# Keep a HAM validation/background dataframe for SHAP background examples.
# This is only used to build a small background tensor set for the binary model.
ham_full_df = pd.read_csv(HAM_CSV).copy()
if "image_exists" in ham_full_df.columns:
    ham_full_df = ham_full_df[ham_full_df["image_exists"].astype(bool)].copy()
ham_full_df["dataset"] = "ham10000"

if "label_name" in ham_full_df.columns:
    binary_map = {"benign": 0, "malignant": 1}
    ham_full_df["binary_label"] = ham_full_df["label_name"].map(binary_map)
    ham_full_df = ham_full_df[ham_full_df["binary_label"].notna()].copy()
    ham_full_df["binary_label"] = ham_full_df["binary_label"].astype(int)

    _, val_df = train_test_split(
        ham_full_df,
        test_size=0.2,
        stratify=ham_full_df["binary_label"],
        random_state=SEED,
    )
else:
    val_df = ham_full_df.copy()

val_df = val_df.reset_index(drop=True)
print("HAM background/validation rows:", len(val_df))


## Transforms and helper functions

In [ ]:
# Validation transform — no augmentation (same as Phase 2)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ImageNet mean/std for denormalisation (needed for display)
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    """Convert normalised tensor → displayable numpy RGB image."""
    img = tensor.detach().cpu().numpy().transpose(1, 2, 0)  # ← add .detach()
    img = (img * STD + MEAN)
    return np.clip(img, 0, 1)

def load_image_tensor(img_path):
    """Load a single image as a (1, C, H, W) tensor on DEVICE."""
    img = Image.open(img_path).convert("RGB")
    return val_transform(img).unsqueeze(0).to(DEVICE)

def predict_prob(tensor):
    """Return malignant probability for a (1,C,H,W) tensor."""
    with torch.no_grad():
        logit = model(tensor)
        return torch.sigmoid(logit).item()

## Sample N images per lesion class

In [ ]:
# Pilot summary
print("Pilot sample counts:")
display(samples_df.groupby("dataset").size().rename("n").reset_index())

if "label_name" in samples_df.columns:
    print("\nHAM diagnostic label distribution inside pilot:")
    display(
        samples_df[samples_df["dataset"] == "ham10000"]
        .groupby(["dx", "label_name"])
        .size()
        .rename("n")
        .reset_index()
    )

print("\nFirst pilot rows:")
display(samples_df[["dataset", "stem", "dx", "label_name"]].head(10))


## Grad-CAM

In [ ]:
from captum.attr import LayerGradCam, LayerAttribution

# Target layer — last conv layer in ResNet-50
target_layer = model.layer4[-1]
gradcam = LayerGradCam(model, target_layer)

def get_gradcam(img_tensor):
    """
    Returns a (224, 224) numpy heatmap for a single (1,C,H,W) tensor.
    target=0 because our model outputs a single logit (binary).
    """
    img_tensor = img_tensor.requires_grad_(True)
    attr = gradcam.attribute(img_tensor, target=0)

    # Upsample to input size
    attr_up = LayerAttribution.interpolate(attr, (IMG_SIZE, IMG_SIZE))
    heatmap = attr_up.squeeze().cpu().detach().numpy()

    # Normalise to [0, 1]
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    return heatmap

# Quick test on first pilot image
test_row = samples_df.iloc[0]
test_path = get_image_path(test_row)
test_t = load_image_tensor(test_path)
heatmap = get_gradcam(test_t)

plt.figure(figsize=(5, 4))
plt.imshow(denormalize(test_t.squeeze(0)))
plt.imshow(heatmap, cmap="jet", alpha=0.45)
plt.title(f"Grad-CAM test — {test_row['dataset']} / {test_row['stem']}")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Grad-CAM working ✓")


## SHAP

In [ ]:
# ── Build a small background set for SHAP ─────────────────────────────────
# Prefer HAM validation images because the model was trained as a HAM binary classifier.
N_SHAP_BACKGROUND = min(50, len(val_df))
bg_rows = val_df.sample(N_SHAP_BACKGROUND, random_state=SEED).copy()

bg_tensors = torch.cat([
    load_image_tensor(HAM_IMG_DIR / f"{row['stem']}.jpg")
    for _, row in bg_rows.iterrows()
], dim=0)  # (N, C, H, W)

def attribution_to_2d_heatmap(x: np.ndarray) -> np.ndarray:
    """Convert common attribution shapes to a normalised 2D heatmap."""
    arr = np.asarray(x)
    arr = np.squeeze(arr)

    # Common SHAP/CNN attribution cases:
    # (C,H,W), (H,W,C), (1,C,H,W), (1,H,W,C)
    if arr.ndim == 4:
        arr = np.squeeze(arr, axis=0) if arr.shape[0] == 1 else arr[0]

    arr = np.squeeze(arr)

    if arr.ndim == 3:
        if arr.shape[0] in (1, 3, 4):       # channel-first
            arr = np.mean(np.abs(arr), axis=0)
        elif arr.shape[-1] in (1, 3, 4):    # channel-last
            arr = np.mean(np.abs(arr), axis=-1)

    if arr.ndim != 2:
        raise ValueError(f"Could not convert attribution to 2D heatmap. Shape: {arr.shape}")

    arr = arr.astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr - arr.min()
    arr = arr / (arr.max() + 1e-8)
    return arr.astype(np.float32)

# Wrapper: SHAP needs a function that takes numpy (N,H,W,C) → numpy (N,1)
def model_predict_shap(images_np):
    """images_np: (N, H, W, C) float32 numpy, pixel values in [0,1]."""
    tensors = []
    for img in images_np:
        t = transforms.ToTensor()(img)
        t = transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])(t)
        tensors.append(t)

    batch = torch.stack(tensors).to(DEVICE)

    with torch.no_grad():
        logits = model(batch)
        probs = torch.sigmoid(logits).cpu().numpy()

    return probs  # (N, 1)

explainer_shap = shap.GradientExplainer(
    model,
    bg_tensors
)

def get_shap(img_tensor):
    """Return a normalised (224, 224) SHAP saliency heatmap."""
    shap_values = explainer_shap.shap_values(img_tensor)
    sv = shap_values[0] if isinstance(shap_values, list) else shap_values
    return attribution_to_2d_heatmap(sv)

# Quick test
sv_map = get_shap(test_t)

plt.figure(figsize=(5, 4))
plt.imshow(denormalize(test_t.squeeze(0)))
plt.imshow(sv_map, cmap="hot", alpha=0.45)
plt.title(f"SHAP test — {test_row['dataset']} / {test_row['stem']}")
plt.axis("off")
plt.tight_layout()
plt.show()

print("SHAP working ✓")


## LIME

In [ ]:
explainer_lime = lime_image.LimeImageExplainer(random_state=SEED)

def get_lime(img_path):
    """
    Returns (explanation, img_np) where img_np is the raw (H,W,3) uint8 array.
    LIME works on raw pixel images, not normalised tensors.
    """
    img_np = np.array(
        Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    )  # uint8 (H, W, 3)

    def batch_predict(images):
        """images: list of (H,W,3) uint8 numpy arrays."""
        return model_predict_shap(
            np.array(images).astype(np.float32) / 255.0
        )

    explanation = explainer_lime.explain_instance(
        img_np,
        batch_predict,
        top_labels=1,
        hide_color=0,
        num_samples=500,    # increase to 1000+ for better quality, slower
        random_seed=SEED
    )
    return explanation, img_np

def lime_heatmap(explanation, img_np):
    """Convert LIME explanation → normalised (H,W) heatmap array."""
    top_label = explanation.top_labels[0]
    # Get positive superpixel contributions
    temp, mask = explanation.get_image_and_mask(
        top_label,
        positive_only=True,
        num_features=10,
        hide_rest=False
    )
    heatmap = explanation.get_image_and_mask(
        top_label,
        positive_only=False,
        num_features=10,
        hide_rest=False
    )[1].astype(np.float32)

    # Use absolute contribution magnitude and normalise to [0, 1].
    heatmap = np.abs(heatmap)
    heatmap = heatmap / (heatmap.max() + 1e-8)
    return heatmap.astype(np.float32)

# Quick test
lime_exp, img_np = get_lime(test_path)
lm = lime_heatmap(lime_exp, img_np)
plt.figure(figsize=(5, 4))
plt.imshow(img_np)
plt.imshow(lm, cmap='RdYlGn', alpha=0.45)
plt.title(f"LIME test — {test_row['dx']} ({test_row['label_name']})")
plt.axis('off')
plt.tight_layout()
plt.show()
print("LIME working ✓")


## Generate and save all maps

In [ ]:
# ── Generate Grad-CAM, SHAP, LIME for every pilot image and save arrays ─────
results = []
manifest_rows = []

print(f"Processing {len(samples_df)} pilot images...\n")

for idx, row in samples_df.iterrows():
    stem = str(row["stem"])
    dataset = str(row["dataset"]).lower()
    safe_id = f"{dataset}_{stem}"

    img_path = get_image_path(row)
    img_tensor = load_image_tensor(img_path)
    prob = predict_prob(img_tensor)

    # Grad-CAM
    gc_map = get_gradcam(img_tensor)

    # SHAP
    sv_map = get_shap(img_tensor)

    # LIME
    lime_exp, img_np = get_lime(img_path)
    lm_map = lime_heatmap(lime_exp, img_np)

    # Save maps as .npy arrays in method-specific folders
    gradcam_path = XAI_OUT_DIR / "gradcam" / f"{safe_id}.npy"
    shap_path = XAI_OUT_DIR / "shap" / f"{safe_id}.npy"
    lime_path = XAI_OUT_DIR / "lime" / f"{safe_id}.npy"

    np.save(gradcam_path, gc_map.astype(np.float32))
    np.save(shap_path, sv_map.astype(np.float32))
    np.save(lime_path, lm_map.astype(np.float32))

    result = {
        "dataset": dataset,
        "stem": stem,
        "safe_id": safe_id,
        "dx": row.get("dx", "unknown"),
        "label_name": row.get("label_name", "unknown"),
        "prob": prob,
        "img_tensor": img_tensor,
        "img_np": img_np,
        "gradcam": gc_map,
        "shap": sv_map,
        "lime": lm_map,
    }
    results.append(result)

    manifest_rows.append({
        "dataset": dataset,
        "stem": stem,
        "safe_id": safe_id,
        "image_path": str(img_path),
        "xai_gradcam_path": str(gradcam_path),
        "xai_shap_path": str(shap_path),
        "xai_lime_path": str(lime_path),
        "model_prob_malignant": prob,
        "dx": row.get("dx", "unknown"),
        "label_name": row.get("label_name", "unknown"),
    })

    print(
        f"  [{idx + 1:03d}/{len(samples_df)}] "
        f"{dataset:8s} | {stem} | p(malignant)={prob:.3f}"
    )

xai_manifest = pd.DataFrame(manifest_rows)
xai_manifest_path = XAI_MANIFEST_DIR / "pilot_xai_manifest.csv"
xai_manifest.to_csv(xai_manifest_path, index=False)

print(f"\nDone. Arrays saved to: {XAI_OUT_DIR.resolve()}")
print(f"Manifest saved to: {xai_manifest_path.resolve()}")

display(xai_manifest.head())


## Visualisation grid (Image | Grad-CAM | SHAP | LIME)

In [ ]:
# ── Pilot visual sanity-check grid ─────────────────────────────────────────
CMAPS = {"gradcam": "jet", "shap": "hot", "lime": "RdYlGn"}

preview_results = results[:12]
n = len(preview_results)

fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
if n == 1:
    axes = np.expand_dims(axes, axis=0)

col_titles = ["Original", "Grad-CAM", "SHAP", "LIME"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight="bold")

for row_i, r in enumerate(preview_results):
    orig = denormalize(r["img_tensor"].squeeze(0))

    axes[row_i, 0].imshow(orig)
    axes[row_i, 0].set_ylabel(
        f"{r['dataset']}\n{r['stem']}\np={r['prob']:.2f}",
        fontsize=8,
        rotation=0,
        labelpad=70,
        va="center",
    )

    for col_i, (method, cmap) in enumerate(CMAPS.items(), start=1):
        axes[row_i, col_i].imshow(orig)
        axes[row_i, col_i].imshow(
            r[method],
            cmap=cmap,
            alpha=0.5,
            vmin=0,
            vmax=1,
        )

    for ax in axes[row_i]:
        ax.axis("off")

plt.tight_layout()
preview_path = XAI_PREVIEW_DIR / "pilot_xai_preview_grid.png"
fig.savefig(preview_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved preview: {preview_path.resolve()}")


## Combined summary figure (report-ready)

In [ ]:
# Optional: one compact mixed-dataset figure for reports/debugging
# This is deliberately not grouped by dx because ISIC has no diagnostic labels here.

report_results = []
for dataset in ["ham10000", "isic2018"]:
    dataset_results = [r for r in results if r["dataset"] == dataset]
    report_results.extend(dataset_results[:3])

if report_results:
    n = len(report_results)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    col_titles = ["Original", "Grad-CAM", "SHAP", "LIME"]
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=12, fontweight="bold")

    for row_i, r in enumerate(report_results):
        orig = denormalize(r["img_tensor"].squeeze(0))

        axes[row_i, 0].imshow(orig)
        axes[row_i, 0].set_ylabel(
            f"{r['dataset']}\n{r['stem']}\np={r['prob']:.2f}",
            fontsize=8,
            rotation=0,
            labelpad=70,
            va="center",
        )

        for col_i, (method, cmap) in enumerate(CMAPS.items(), start=1):
            axes[row_i, col_i].imshow(orig)
            axes[row_i, col_i].imshow(r[method], cmap=cmap, alpha=0.5, vmin=0, vmax=1)

        for ax in axes[row_i]:
            ax.axis("off")

    plt.tight_layout()
    report_path = XAI_PREVIEW_DIR / "pilot_xai_report_examples.png"
    fig.savefig(report_path, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"Saved report examples: {report_path.resolve()}")
else:
    print("No results available for report figure.")
